# Transverse geometry and background contributions

Separate the FAC-related mean rotation from directional anisotropy, verify
simple analytic maps, and attribute magnetic gradients before taking norms.
Definitions follow the [baseline theory](../../docs/fac_anisotropy_theory.md)
([日本語訳](../../docs/fac_anisotropy_theory_ja.md)); R1/R2 interpretations there
remain research hypotheses, not classifications implemented by this notebook.

Run top to bottom in a fresh Python kernel after installing `.[examples]` from
    the repository root. No external data, network access, Qt or PyVista is required.
    Stored outputs are cleared; figures and assertions are generated by the cells.
    [Notebook index](README.md) · [Analysis guide](../../docs/geometry_analysis.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mageometry.geometry import field_line_transverse_geometry, field_line_transverse_decomposition

def linear_field(a=0., q=-1., p=1., d=0.):
    # At the origin T=z, n=x, b=y and curvature=1.
    def field(x, y, z):
        x, y, z = np.broadcast_arrays(x, y, z)
        return a*x + q*y + z, p*x + d*y, np.ones_like(z)
    return field

## The same FAC can accompany different shear

Rows of M = [[a, q], [p, d]] denote output components in (n, b), columns derivative
directions. Here q denotes only that matrix entry. The canonical diagnostics are
alpha = p−q, beta_g = p+q, delta_g = a−d, gamma = √(beta_g²+delta_g²).
Eta = (alpha²−gamma²)/(alpha²+gamma²), undefined when both vanish.

The added z term in Bx gives a resolved curvature normal at the origin without
changing M. These are local illustrative fields, not global magnetospheric models.

In [ ]:
examples = {
    'Rotation': linear_field(p=1., q=-1.),
    'Simple shear': linear_field(p=2., q=0.),
    'Symmetric shear': linear_field(p=1., q=1.),
    'Axis strain': linear_field(a=1., d=-1., p=0., q=0.),
}
results = {name: field_line_transverse_geometry(field, 0., 0., 0., delta=.001)
           for name, field in examples.items()}
np.testing.assert_allclose([r['eta'] for r in results.values()], [1., 0., -1., -1.])
np.testing.assert_allclose(results['Rotation']['alpha'], results['Simple shear']['alpha'])
assert results['Axis strain']['beta_g'] == 0 and results['Axis strain']['gamma'] > 0
for name, result in results.items():
    print(name, {key: result[key] for key in ('alpha', 'beta_g', 'delta_g', 'gamma', 'eta')})

## Directional rotation is not its mean

For a transverse separation at angle φ from n,

$$\Omega(\phi)=\alpha/2+(\beta_g/2)\cos 2\phi-(\delta_g/2)\sin 2\phi.$$

The azimuthal mean is alpha/2 and extrema are (alpha ± gamma)/2.
The Frenet-frame angle obeys dφ/ds = Ω−τ; do not omit the frame torsion.
Local rotational behavior or positive eta does not establish finite-distance winding.

In [ ]:
phi = np.linspace(0., 2*np.pi, 200, endpoint=False)
fig, ax = plt.subplots()
for name, result in results.items():
    omega = (result['alpha'] + result['beta_g']*np.cos(2*phi)
             - result['delta_g']*np.sin(2*phi))/2
    np.testing.assert_allclose(omega.mean(), result['alpha']/2, atol=1e-14)
    ax.plot(phi, omega, label=name)
ax.set(xlabel='Separation angle [rad]', ylabel='Angular rate [1/length]')
ax.legend()
plt.show()

## Background attribution fixes the total-field frame

All branches use the total field's magnitude, tangent and transverse projector.
Subtract gradients first: G_residual = G_total − G_background. Linear quantities
add, but gamma and eta do not. The shear cross term can reinforce or cancel.
The background below has no curl but has symmetric transverse strain.

In [ ]:
total = linear_field(a=.5, d=-.5, p=2., q=-1.)
background = lambda x, y, z: (x, -y, 0.*z)
parts = field_line_transverse_decomposition(total, background, 0., 0., 0., delta=.001)
t, bg, residual = (parts[key] for key in ('total', 'background', 'residual'))
for key in ('gradient', 'transverse', 'shear', 'alpha', 'beta_g', 'delta_g'):
    np.testing.assert_allclose(t[key], bg[key] + residual[key], atol=1e-12)
cross_term = 4*np.trace(bg['shear'] @ residual['shear'])
np.testing.assert_allclose(t['gamma']**2, bg['gamma']**2 + residual['gamma']**2 + cross_term)
assert not np.isclose(residual['gamma'], t['gamma'] - bg['gamma'])
print('gamma total/background/residual:', t['gamma'], bg['gamma'], residual['gamma'])
print('Shear cross term:', cross_term)

## Undefined ratios and straight fields

A zero residual gradient has zero alpha/gamma and undefined eta. Only beta_g and
delta_g among the six diagnostics require a curvature normal. Contribution eta
and omega_c describe projected operators, not the field-line winding of the
background or residual field taken separately.

In [ ]:
zero = field_line_transverse_decomposition(total, total, 0., 0., 0.)['residual']
assert zero['alpha'] == 0 and zero['gamma'] == 0 and np.isnan(zero['eta'])
straight = lambda x, y, z: (-y, x, 1. + 0.*z)
rates = field_line_transverse_geometry(straight, 0., 0., 0.)
assert np.isfinite(rates['alpha']) and np.isfinite(rates['gamma'])
assert np.isnan(rates['beta_g']) and np.isnan(rates['delta_g'])

For models or simulation data, repeat these calculations on broadcast coordinates,
using consistent input units and checking derivative/grid convergence. Background
attribution does not remove background dependence of the total-field frame and
does not automatically identify a physical current source. See the
[transverse API guide](../../docs/transverse_geometry.md#background-gradient-contributions)
for model and viewer examples. The theory and concept figures provide the full
mathematical derivation; this notebook focuses on executable comparisons.